<a href="https://colab.research.google.com/github/heldo07/3709-nodejs-lib-arquivos-iniciais/blob/main/Frequencia_de_alunos_cp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from google.colab import files

print("1. Envie a planilha de CATRACA (relatório de acessos):")
uploaded_catraca = files.upload()
nome_arq_catraca = list(uploaded_catraca.keys())[0]

print("\n2. Envie a planilha de CALENDÁRIO (dias esperados por turma):")
uploaded_calendario = files.upload()
nome_arq_calendario = list(uploaded_calendario.keys())[0]

# Leitura inicial para você conferir os nomes das colunas se precisar
df_catraca = pd.read_excel(nome_arq_catraca)
df_calendario = pd.read_excel(nome_arq_calendario)

print("\n--- Arquivos carregados com sucesso! ---")

1. Envie a planilha de CATRACA (relatório de acessos):


Saving IKAcesso_Frequência_2026-09-22.xlsx to IKAcesso_Frequência_2026-09-22 (1).xlsx

2. Envie a planilha de CALENDÁRIO (dias esperados por turma):


KeyboardInterrupt: 

In [3]:
# AJUSTE AQUI OS NOMES EXATOS DAS COLUNAS DA SUA CATRACA:
COL_MATRICULA = "matricula"
COL_NOME = "nome"
COL_TURMA = "turma"
COL_DATA_HORA = "data_hora"
# =========================================================

# 1. Tratar a catraca (pegar apenas a data e remover duplicadas no mesmo dia)
df_catraca["data"] = pd.to_datetime(df_catraca[COL_DATA_HORA]).dt.date

df_presenca_dias = (
    df_catraca[[COL_MATRICULA, COL_NOME, COL_TURMA, "data"]]
    .drop_duplicates()
    .groupby([COL_MATRICULA, COL_NOME, COL_TURMA])
    .size()
    .reset_index(name="dias_presente")
)

# Renomear para padronizar
df_presenca_dias.columns = ["matricula", "nome", "turma", "dias_presente"]

# 2. Cruzar com a planilha de dias esperados por turma
df_final = pd.merge(df_presenca_dias, df_calendario, on="turma", how="left")

# 3. Calcular frequência e aplicar regras do Pé-de-Meia
FREQ_MINIMA_PCT = 80.0

df_final["perc_frequencia"] = (
    df_final["dias_presente"] / df_final["dias_total_esperado"]
) * 100
df_final["status_pe_de_meia"] = df_final["perc_frequencia"].apply(
    lambda x: "Habilitado" if x >= FREQ_MINIMA_PCT else "Não Habilitado"
)
df_final["alerta_risco"] = (
    df_final["perc_frequencia"] >= FREQ_MINIMA_PCT - 5
) & (df_final["perc_frequencia"] < FREQ_MINIMA_PCT)

print("Cálculos realizados com sucesso!")
print(df_final.head())

KeyError: 'data_hora'

In [5]:
# 1. Mostrar um resumo geral de quantos passaram e quantos ficaram de fora
print("=== RESUMO DE ELEGIBILIDADE ===")
print(df_final["status_pe_de_meia"].value_counts())
print("-" * 30)

# 2. Filtrar e exibir apenas os alunos NÃO HABILITADOS para conferência rápida
df_nao_habilitados = df_final[df_final["status_pe_de_meia"] == "Não Habilitado"]

print(
    f"\n⚠️ Alunos Abaixo do Limite (Total: {len(df_nao_habilitados)} alunos):"
)
# Exibe as colunas principais de forma limpa na tela do Colab
display(
    df_nao_habilitados[
        [
            "matricula",
            "nome",
            "turma",
            "dias_presente",
            "dias_total_esperado",
            "perc_frequencia",
        ]
    ].reset_index(drop=True)
)

# 3. Salvar e baixar o arquivo Excel completo
arquivo_saida = "resultado_frequencia_pedemeia.xlsx"
df_final.to_excel(arquivo_saida, index=False)

print(f"\n📥 Baixando o relatório completo: {arquivo_saida}")
files.download(arquivo_saida)

=== RESUMO DE ELEGIBILIDADE ===


NameError: name 'df_final' is not defined